<a href="https://colab.research.google.com/github/DimiGretsistas/Automated-Customers-Reviews-Project/blob/main/notebooks/04_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Review Summarization

**Goals**
- Use a small generative model to create short recommendation articles from clustered product reviews.
- This notebook recreates the clustered sample directly, so it does not depend on the clustering notebook.

In [3]:
import pandas as pd

# load cleaned dataset from Colab
df = pd.read_csv(
    "/content/clean_reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

# inspect data
df.head()

,name,reviews.rating,reviews.text,sentiment,clean_review
0,AmazonBasics AAA Performance Alkaline Batterie...,3,I order 3 of them and one of the item is bad q...,neutral,i order 3 of them and one of the item is bad q...
1,AmazonBasics AAA Performance Alkaline Batterie...,4,Bulk is always the less expensive way to go fo...,positive,bulk is always the less expensive way to go fo...
2,AmazonBasics AAA Performance Alkaline Batterie...,5,Well they are not Duracell but for the price i...,positive,well they are not duracell but for the price i...
3,AmazonBasics AAA Performance Alkaline Batterie...,5,Seem to work as well as name brand batteries a...,positive,seem to work as well as name brand batteries a...
4,AmazonBasics AAA Performance Alkaline Batterie...,5,These batteries are very long lasting the pric...,positive,these batteries are very long lasting the pric...


## Create Sample for Summarization
- Use a smaller sample to keep the notebook fast.

In [4]:
# use a smaller sample for fast processing
sample_df = df.sample(n=1000, random_state=42).copy()

# check sample size
sample_df.shape

(1000, 5)

## Create Embeddings and Clusters
- Use a pretrained sentence transformer and KMeans to recreate product clusters.

In [5]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# prepare review texts
texts = sample_df["clean_review"].tolist()

# load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# create embeddings
embeddings = embedding_model.encode(texts, show_progress_bar=True)

# create 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42)

# assign clusters
sample_df["cluster"] = kmeans.fit_predict(embeddings)

sample_df[["name", "clean_review", "cluster"]].head()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

,name,clean_review,cluster
35736,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",this is the 2nd one we have purchased for our ...,1
24525,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",it takes forever to download apps i m returnin...,3
49321,"Echo (White),,,\r\nEcho (White),,,",excellent product tones of fun and super usabi...,1
12351,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",easy to use lots of books obviously but also l...,2
31628,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",good to have for smaller tablet use lightweigh...,3


## Inspect Clustered Sample
- Check available clusters and products in the sampled data.

In [6]:
# check sample dataset size
print(sample_df.shape)

# check available clusters
print(sample_df["cluster"].value_counts())

# inspect clustered sample
sample_df[["name", "reviews.rating", "sentiment", "cluster"]].head()

(1000, 6)
cluster
3    303
1    290
2    161
4    141
0    105
Name: count, dtype: int64


,name,reviews.rating,sentiment,cluster
35736,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",5,positive,1
24525,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",1,negative,3
49321,"Echo (White),,,\r\nEcho (White),,,",5,positive,1
12351,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",5,positive,2
31628,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",4,positive,3


## Prepare Product Insights
- Extract top products, worst product, and complaints per cluster.

In [7]:
def prepare_cluster_insights(cluster_df):
    # calculate average rating per product
    product_stats = cluster_df.groupby("name").agg(
        avg_rating=("reviews.rating", "mean"),
        review_count=("reviews.rating", "count")
    ).reset_index()

    # keep products with at least 3 reviews
    product_stats = product_stats[product_stats["review_count"] >= 3]

    # top 3 products
    top_products = product_stats.sort_values(
        by="avg_rating",
        ascending=False
    ).head(3)

    # worst product
    worst_product = product_stats.sort_values(
        by="avg_rating",
        ascending=True
    ).head(1)

    # complaints (negative reviews)
    complaints = cluster_df[
        cluster_df["sentiment"] == "negative"
    ]["clean_review"].head(5).tolist()

    return top_products, worst_product, complaints

In [8]:
# test for one cluster
cluster_df = sample_df[sample_df["cluster"] == 0]

top_products, worst_product, complaints = prepare_cluster_insights(cluster_df)

print("Top products:")
print(top_products)

print("\nWorst product:")
print(worst_product)

print("\nComplaints:")
print(complaints)

Top products:
                                                name  avg_rating  review_count
4             Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,    4.629630            27
9                 Echo (White),,,\r\nEcho (White),,,    4.583333            36
3  Amazon Echo Show Alexa-enabled Bluetooth Speak...    4.500000            14

Worst product:
                                          name  avg_rating  review_count
2  Amazon - Echo Plus w/ Built-In Hub - Silver         4.1            10

Complaints:
['it was a bit more involved getting set up but thereafter the function is simply talking to alexia the plug adapter to turn on and off lights tv is useful and certainly worth having', 'we returned it could not give answers to our questions even though the answers were easily found on google', 'i purchased this item on the recommendation of many existing owners however when i got home and attempted to set up the device it was frustrating and the instructions faqs were not very helpful the e

##Clean Product Names
- Fix formatting issues in product names.

In [15]:
# clean messy product names (remove commas, line breaks)
sample_df["name"] = sample_df["name"].str.replace(r"[\r\n,]+", " ", regex=True).str.strip()

##-----------------------------------------------------------
## Inspect Clustered Sample
- Check available clusters and products in the sampled data.

In [16]:
# check sample dataset size
print(sample_df.shape)

# check available clusters
print(sample_df["cluster"].value_counts())

# inspect clustered sample
sample_df[["name", "reviews.rating", "sentiment", "cluster"]].head()


(1000, 6)
cluster
3    303
1    290
2    161
4    141
0    105
Name: count, dtype: int64


,name,reviews.rating,sentiment,cluster
35736,Fire Tablet 7 Display Wi-Fi 8 GB - Includes...,5,positive,1
24525,Fire Kids Edition Tablet 7 Display Wi-Fi 16...,1,negative,3
49321,Echo (White) Echo (White),5,positive,1
12351,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi ...,5,positive,2
31628,Fire Tablet 7 Display Wi-Fi 8 GB - Includes...,4,positive,3


##Load Generator Model
- Use a small generative model to create recommendation text.

In [17]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=300,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

##Generate Recommendation Article
- Create a short blog-style summary for one cluster.

In [18]:
# select cluster
cluster_id = 0

cluster_df = sample_df[sample_df["cluster"] == cluster_id]

top_products, worst_product, complaints = prepare_cluster_insights(cluster_df)

prompt = f"""
Write a short product recommendation article.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Complaints:
{complaints}

Include:
- Top 3 products
- Differences
- Complaints
- Worst product warning
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Write a short product recommendation article.

Top products:
                                                           name  avg_rating  review_count
                                  Amazon Fire Tv Amazon Fire Tv    4.629630            27
                                      Echo (White) Echo (White)    4.583333            36
Amazon Echo Show Alexa-enabled Bluetooth Speaker with 7" Screen    4.500000            14

Worst product:
                                       name  avg_rating  review_count
Amazon - Echo Plus w/ Built-In Hub - Silver         4.1            10

Complaints:
['it was a bit more involved getting set up but thereafter the function is simply talking to alexia the plug adapter to turn on and off lights tv is useful and certainly worth having', 'we returned it could not give answers to our questions even though the answers were easily found on google', 'i purchased this item on the recommendation of many existing owners however when i got home and attempted to set 

In [19]:
#Prompt Variation 2

prompt = f"""
You are a tech reviewer.

Write a short, clear product recommendation article based ONLY on the data below.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- Be concise and factual
- Do NOT invent products
- Do NOT mention brands not listed
- Focus on differences and real issues

Structure:
1. Introduction
2. Top products comparison
3. Key complaints
4. Worst product warning
5. Final recommendation
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a tech reviewer.

Write a short, clear product recommendation article based ONLY on the data below.

Top products:
                                                           name  avg_rating  review_count
                                  Amazon Fire Tv Amazon Fire Tv    4.629630            27
                                      Echo (White) Echo (White)    4.583333            36
Amazon Echo Show Alexa-enabled Bluetooth Speaker with 7" Screen    4.500000            14

Worst product:
                                       name  avg_rating  review_count
Amazon - Echo Plus w/ Built-In Hub - Silver         4.1            10

Customer complaints:
['it was a bit more involved getting set up but thereafter the function is simply talking to alexia the plug adapter to turn on and off lights tv is useful and certainly worth having', 'we returned it could not give answers to our questions even though the answers were easily found on google', 'i purchased this item on the recommendatio

In [20]:
#Prompt variation 3

prompt = f"""
You are a tech reviewer.

Write a short, structured product recommendation article based ONLY on the data below.

Top products:
{top_products.to_string(index=False)}

Worst product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- Only use the information provided
- Do NOT invent features or products
- Be concise and factual

Structure:
1. Introduction
2. Top products comparison (bullet points)
3. Key complaints (bullet points)
4. Worst product warning
5. Final recommendation
"""

result = generator(prompt)

print(result[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a tech reviewer.

Write a short, structured product recommendation article based ONLY on the data below.

Top products:
                                                           name  avg_rating  review_count
                                  Amazon Fire Tv Amazon Fire Tv    4.629630            27
                                      Echo (White) Echo (White)    4.583333            36
Amazon Echo Show Alexa-enabled Bluetooth Speaker with 7" Screen    4.500000            14

Worst product:
                                       name  avg_rating  review_count
Amazon - Echo Plus w/ Built-In Hub - Silver         4.1            10

Customer complaints:
['it was a bit more involved getting set up but thereafter the function is simply talking to alexia the plug adapter to turn on and off lights tv is useful and certainly worth having', 'we returned it could not give answers to our questions even though the answers were easily found on google', 'i purchased this item on the recommen

##Final Conclusions

- The project successfully demonstrates an end-to-end NLP pipeline for analyzing customer reviews.

- The sentiment classifier (DistilBERT) achieved high performance (~95% accuracy), but results are influenced by dataset imbalance, leading to weaker detection of neutral reviews.

- The clustering approach (MiniLM embeddings + KMeans) revealed meaningful product groupings, although quantitative evaluation (low silhouette score) shows that text clustering remains challenging.

- The summarization model (Qwen 1.5B) was able to generate structured recommendation articles using extracted insights such as top products and customer complaints.

- Prompt design proved to be critical for controlling output quality and reducing hallucinations in generative models.

- Overall, combining classification, clustering, and generation provides valuable business insights, transforming raw reviews into actionable recommendations.

- Limitations include dataset imbalance, noisy text data, and the inherent challenges of unsupervised clustering and generative AI reliability.